# Machine Learning Boilerplate

Customizable template for classical ML pipelines (classification and regression).
**Edit the Config cell below, then Run All.** No other cell needs to change.

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif, f_regression
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor, RandomForestRegressor
from sklearn.svm import SVC, SVR
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LinearRegression, Lasso, Ridge
import xgboost as xgb

from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    mean_squared_error, mean_absolute_error, r2_score
)

---
## Config

**This is the only cell you normally need to edit.**

| Parameter | Description |
|-----------|-------------|
| `FILE_PATH` | Path to CSV file containing the dataset |
| `TARGET_COL` | Name of the target column |
| `TASK` | `"classification"` or `"regression"` |
| `MODEL` | Key from the model registry (see next cell) |
| `REDUCTION` | `"none"`, `"pca"`, or `"kbest"` |
| `NUM_FEATURES` | Number of features to select (kbest only) |
| `VARIANCE` | Variance to retain (pca only, 0–1) |
| `TRAIN_SIZE` | Fraction of data for training |
| `NUM_FOLD` | Cross-validation folds |
| `RANDOM_SEED` | Seed for reproducibility |
| `SAVE_EXPERIMENT` | Save config, params, metrics and plots to `experiments/` |

In [ ]:
# ── dataset ──────────────────────────────────────────────────────────────────
FILE_PATH  = "data/data.csv"   # path to your CSV file
TARGET_COL = "target"          # column to predict

# ── task & model ─────────────────────────────────────────────────────────────
TASK  = "classification"       # "classification" | "regression"
MODEL = "random_forest"        # key from MODELS registry (cell below)

# ── preprocessing ────────────────────────────────────────────────────────────
ENCODE_CATEGORICAL = True
REDUCTION    = "none"          # "none" | "pca" | "kbest"
NUM_FEATURES = 10              # used when REDUCTION = "kbest"
VARIANCE     = 0.95            # used when REDUCTION = "pca"

# ── evaluation ───────────────────────────────────────────────────────────────
TRAIN_SIZE = 0.8
NUM_FOLD   = 5

# ── reproducibility ──────────────────────────────────────────────────────────
RANDOM_SEED = 42

# ── experiment tracking ──────────────────────────────────────────────────────
SAVE_EXPERIMENT = True         # write config, params, metrics, plots to experiments/

## Model Registry

Each entry: `"key": (estimator, grid_params)`.

Set `MODEL` in Config to pick one. Add or modify entries freely — grid params
are prefixed with `model__` to match the pipeline step name.

In [ ]:
MODELS = {
    # ── classification ───────────────────────────────────────────────────────
    "random_forest": (
        RandomForestClassifier(random_state=RANDOM_SEED),
        {
            "model__n_estimators": [50, 100, 200],
            "model__max_depth":    [5, 10, None],
        },
    ),
    "svc": (
        SVC(random_state=RANDOM_SEED),
        {
            "model__C":      [0.1, 1, 10],
            "model__kernel": ["linear", "rbf"],
            "model__gamma":  ["scale", "auto"],
        },
    ),
    "xgboost": (
        xgb.XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss"),
        {
            "model__n_estimators":  [50, 100, 200],
            "model__learning_rate": [0.01, 0.1, 0.3],
            "model__max_depth":     [3, 5, 7],
        },
    ),
    "mlp": (
        MLPClassifier(random_state=RANDOM_SEED, max_iter=500),
        {
            "model__hidden_layer_sizes": [(50,), (100,), (50, 50)],
            "model__activation":         ["relu", "tanh"],
            "model__alpha":              [0.0001, 0.001],
        },
    ),
    # ── regression ───────────────────────────────────────────────────────────
    "linear": (
        LinearRegression(),
        {
            "model__fit_intercept": [True, False],
        },
    ),
    "ridge": (
        Ridge(),
        {
            "model__alpha":    [0.1, 1.0, 10.0],
            "model__max_iter": [1000, 5000],
        },
    ),
    "lasso": (
        Lasso(random_state=RANDOM_SEED),
        {
            "model__alpha":    [0.1, 1.0, 10.0],
            "model__max_iter": [1000, 5000],
        },
    ),
    "rf_regressor": (
        RandomForestRegressor(random_state=RANDOM_SEED),
        {
            "model__n_estimators":   [50, 100, 200],
            "model__max_depth":      [5, 10, None],
            "model__min_samples_split": [2, 5, 10],
        },
    ),
    "svr": (
        SVR(),
        {
            "model__C":      [0.1, 1, 10],
            "model__kernel": ["linear", "rbf"],
            "model__gamma":  ["scale", "auto"],
        },
    ),
    "gbr": (
        GradientBoostingRegressor(random_state=RANDOM_SEED),
        {
            "model__n_estimators":  [50, 100, 200],
            "model__learning_rate": [0.01, 0.1, 0.3],
            "model__max_depth":     [3, 5, 7],
        },
    ),
}

assert MODEL in MODELS, f"MODEL '{MODEL}' not found. Available: {list(MODELS.keys())}"
model, grid_params = MODELS[MODEL]
print(f"Selected model: {MODEL} → {model.__class__.__name__}")

---
<center><h1>Dataset</h1></center>

## Load Dataset

Loads CSV from `FILE_PATH` and splits into features `X` and target `y`.
If your data has no ground-truth column yet, build `y` here before the split.

In [ ]:
try:
    data = pd.read_csv(FILE_PATH)
    print(f"Loaded {FILE_PATH} — shape: {data.shape}")
except FileNotFoundError:
    raise FileNotFoundError(f"File not found: {FILE_PATH}")

try:
    X = data.drop(columns=[TARGET_COL])
    y = data[TARGET_COL]
    print(f"X: {X.shape}  |  y: {y.shape}")
except KeyError:
    raise KeyError(f"Target column '{TARGET_COL}' not found in dataset.")

## EDA Plots

Target distribution and feature correlation heatmap.

In [ ]:
plt.figure(figsize=(8, 5))

if TASK == "classification":
    order = y.value_counts().index
    ax = sns.countplot(x=y, order=order, hue=y, palette="viridis", legend=False)
    for p in ax.patches:
        ax.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2, p.get_height()),
                    ha="center", va="bottom", fontsize=11)
    plt.title("Class Distribution", fontsize=16)
    plt.xlabel("Class", fontsize=13)
    plt.ylabel("Count", fontsize=13)
else:
    sns.histplot(y, kde=True, color="steelblue")
    plt.title("Target Distribution", fontsize=16)
    plt.xlabel(TARGET_COL, fontsize=13)
    plt.ylabel("Frequency", fontsize=13)

plt.xticks(rotation=45, fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
numeric_X = X.select_dtypes(include="number")
if numeric_X.shape[1] > 1:
    plt.figure(figsize=(min(12, numeric_X.shape[1] + 2), min(10, numeric_X.shape[1])))
    sns.heatmap(numeric_X.corr(), annot=True, cmap="coolwarm", center=0,
                fmt=".2f", square=True, linewidths=0.5)
    plt.title("Correlation Heatmap", fontsize=16)
    plt.tight_layout()
    plt.show()
else:
    print("Not enough numeric features for a correlation heatmap.")

---
<center><h1>Preprocessing</h1></center>

## Encode Categorical Features

One-hot encodes categorical columns in `X`. For classification tasks, encodes
the target `y` with `LabelEncoder` (produces 1-d integer labels compatible with
all classifiers and `stratify=y`). Toggle with `ENCODE_CATEGORICAL`.

In [ ]:
if ENCODE_CATEGORICAL:
    cat_cols = X.select_dtypes(include=["object", "category"]).columns
    if len(cat_cols) > 0:
        enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        X_enc = pd.DataFrame(
            enc.fit_transform(X[cat_cols]),
            columns=enc.get_feature_names_out(cat_cols),
            index=X.index,
        )
        X = pd.concat([X.drop(columns=cat_cols), X_enc], axis=1)
        print(f"One-hot encoded {len(cat_cols)} categorical column(s). X shape: {X.shape}")
    else:
        print("No categorical columns found in X.")

    if TASK == "classification" and y.dtype in ["object", "category"]:
        le = LabelEncoder()
        y = pd.Series(le.fit_transform(y), index=y.index, name=TARGET_COL)
        print(f"Label-encoded target. Classes: {list(le.classes_)}")

## Handle Missing Values

Strategy based on column type and percentage missing:

| Missing % | Numeric | Categorical |
|-----------|---------|-------------|
| 0% | — | — |
| > 70% | drop column | drop column |
| 5–70% | median + indicator | `"Missing"` + indicator |
| < 5% | median | mode |

In [ ]:
missing_stats = pd.DataFrame({
    "Total Missing":   X.isnull().sum(),
    "Percent Missing": (X.isnull().sum() / len(X) * 100).round(2),
})
n_missing = (missing_stats["Total Missing"] > 0).sum()

if n_missing == 0:
    print("No missing values.")
else:
    print(f"Missing value summary ({n_missing} column(s)):")
    print(missing_stats[missing_stats["Total Missing"] > 0].sort_values(
        "Percent Missing", ascending=False))
    print(f"\nShape before: {X.shape}")

    for col in list(X.columns):
        pct = X[col].isnull().mean() * 100
        if pct == 0:
            continue
        elif pct > 70:
            X.drop(columns=[col], inplace=True)
            print(f"Dropped '{col}' ({pct:.1f}% missing)")
        elif pd.api.types.is_numeric_dtype(X[col]):
            if pct < 5:
                X[col] = X[col].fillna(X[col].median())
                print(f"Imputed '{col}' with median")
            else:
                X[f"{col}_missing"] = X[col].isnull().astype(int)
                X[col] = X[col].fillna(X[col].median())
                print(f"Imputed '{col}' with median + added indicator")
        else:
            if pct < 5:
                X[col] = X[col].fillna(X[col].mode()[0])
                print(f"Imputed '{col}' with mode")
            else:
                X[f"{col}_missing"] = X[col].isnull().astype(int)
                X[col] = X[col].fillna("Missing")
                print(f"Imputed '{col}' with 'Missing' + added indicator")

    print(f"Shape after:  {X.shape}")

assert X.isnull().sum().sum() == 0, "Missing values still present after imputation."

## Build Pipeline

Assembles: `StandardScaler → [optional reduction] → model`.

Reduction is driven by `REDUCTION` in Config:
- `"none"` — scale only
- `"pca"` — retain `VARIANCE` of explained variance
- `"kbest"` — select top `NUM_FEATURES` using the appropriate score function for the task

In [ ]:
steps = [("scaler", StandardScaler())]

if REDUCTION == "pca":
    steps.append(("reduce", PCA(n_components=VARIANCE)))
elif REDUCTION == "kbest":
    score_func = f_classif if TASK == "classification" else f_regression
    steps.append(("reduce", SelectKBest(score_func, k=NUM_FEATURES)))
elif REDUCTION != "none":
    raise ValueError(f"REDUCTION must be 'none', 'pca', or 'kbest'. Got: '{REDUCTION}'")

steps.append(("model", model))
pipeline = Pipeline(steps)

print("Pipeline steps:", " → ".join(name for name, _ in steps))

---
<center><h1>Train & Evaluate</h1></center>

## Train-Test Split & Hyperparameter Tuning

Splits data (stratified for classification), then runs `GridSearchCV` with
task-appropriate scoring. Prints per-fold mean ± std for each parameter set.

In [ ]:
stratify = y if TASK == "classification" else None
scoring  = "accuracy" if TASK == "classification" else "neg_mean_squared_error"

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=TRAIN_SIZE, stratify=stratify, random_state=RANDOM_SEED
)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=grid_params,
    scoring=scoring,
    cv=NUM_FOLD,
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)
best_pipeline = grid_search.best_estimator_

cv_results = pd.DataFrame(grid_search.cv_results_)
sign = 1 if TASK == "classification" else -1
print("\nCross-Validation Results:")
for i, row in cv_results.iterrows():
    mean = sign * row["mean_test_score"]
    std  = row["std_test_score"]
    metric = "Accuracy" if TASK == "classification" else "MSE"
    print(f"  Set {i+1:>2}: {metric} = {mean:.4f} ± {std:.4f}  {row['params']}")

print(f"\nBest params: {grid_search.best_params_}")

## Evaluation on Test Set

Evaluates the best pipeline on held-out test data.

- **Classification**: accuracy, classification report, confusion matrix.
- **Regression**: MSE, MAE, R².

In [ ]:
y_pred = best_pipeline.predict(X_test)

if TASK == "classification":
    acc    = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)
    cm     = confusion_matrix(y_test, y_pred)

    print(f"Test Accuracy: {acc:.4f}")
    print("\nClassification Report:")
    print(report)

    labels = sorted(y.unique())
    plt.figure(figsize=(max(5, len(labels)), max(4, len(labels))))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=labels, yticklabels=labels)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.show()

else:
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2  = r2_score(y_test, y_pred)

    print(f"Test MSE : {mse:.4f}")
    print(f"Test MAE : {mae:.4f}")
    print(f"Test R²  : {r2:.4f}")

    residuals = y_test.values - y_pred
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].scatter(y_pred, residuals, alpha=0.5, edgecolors="k", linewidths=0.3)
    axes[0].axhline(0, color="red", linewidth=1)
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("Residual")
    axes[0].set_title("Residuals vs Predicted")
    axes[1].scatter(y_test, y_pred, alpha=0.5, edgecolors="k", linewidths=0.3)
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    axes[1].plot(lims, lims, "r--", linewidth=1)
    axes[1].set_xlabel("True")
    axes[1].set_ylabel("Predicted")
    axes[1].set_title("Predicted vs True")
    plt.tight_layout()
    plt.show()

---
## Save Experiment *(optional)*

When `SAVE_EXPERIMENT = True`, writes a timestamped folder under `experiments/`:

```
experiments/<YYYY-MM-DD_HH-MM-SS>/
  config.txt          resolved config values
  best_params.txt     best hyperparameters from grid search
  metrics.txt         test metrics
  plots/              evaluation figures
```

In [ ]:
if SAVE_EXPERIMENT:
    from pathlib import Path
    from datetime import datetime

    ts      = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    exp_dir = Path("experiments") / ts
    (exp_dir / "plots").mkdir(parents=True, exist_ok=True)

    # config.txt
    cfg_lines = [
        f"FILE_PATH          = {FILE_PATH}",
        f"TARGET_COL         = {TARGET_COL}",
        f"TASK               = {TASK}",
        f"MODEL              = {MODEL}",
        f"ENCODE_CATEGORICAL = {ENCODE_CATEGORICAL}",
        f"REDUCTION          = {REDUCTION}",
        f"NUM_FEATURES       = {NUM_FEATURES}",
        f"VARIANCE           = {VARIANCE}",
        f"TRAIN_SIZE         = {TRAIN_SIZE}",
        f"NUM_FOLD           = {NUM_FOLD}",
        f"RANDOM_SEED        = {RANDOM_SEED}",
    ]
    (exp_dir / "config.txt").write_text("\n".join(cfg_lines))

    # best_params.txt
    (exp_dir / "best_params.txt").write_text(str(grid_search.best_params_))

    # metrics.txt
    if TASK == "classification":
        metrics_text = f"Accuracy: {acc:.4f}\n\n{report}"
    else:
        metrics_text = f"MSE: {mse:.4f}\nMAE: {mae:.4f}\nR2:  {r2:.4f}"
    (exp_dir / "metrics.txt").write_text(metrics_text)

    # plots
    if TASK == "classification":
        fig, ax = plt.subplots(figsize=(max(5, len(labels)), max(4, len(labels))))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                    xticklabels=labels, yticklabels=labels, ax=ax)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        ax.set_title("Confusion Matrix")
        fig.tight_layout()
        fig.savefig(exp_dir / "plots" / "confusion_matrix.png", dpi=150)
        plt.close(fig)
    else:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].scatter(y_pred, residuals, alpha=0.5, edgecolors="k", linewidths=0.3)
        axes[0].axhline(0, color="red", linewidth=1)
        axes[0].set_xlabel("Predicted")
        axes[0].set_ylabel("Residual")
        axes[0].set_title("Residuals vs Predicted")
        axes[1].scatter(y_test, y_pred, alpha=0.5, edgecolors="k", linewidths=0.3)
        axes[1].plot(lims, lims, "r--", linewidth=1)
        axes[1].set_xlabel("True")
        axes[1].set_ylabel("Predicted")
        axes[1].set_title("Predicted vs True")
        fig.tight_layout()
        fig.savefig(exp_dir / "plots" / "regression_plots.png", dpi=150)
        plt.close(fig)

    # correlation heatmap
    if numeric_X.shape[1] > 1:
        fig, ax = plt.subplots(figsize=(min(12, numeric_X.shape[1] + 2),
                                        min(10, numeric_X.shape[1])))
        sns.heatmap(numeric_X.corr(), annot=True, cmap="coolwarm", center=0,
                    fmt=".2f", square=True, linewidths=0.5, ax=ax)
        ax.set_title("Correlation Heatmap")
        fig.tight_layout()
        fig.savefig(exp_dir / "plots" / "correlation_heatmap.png", dpi=150)
        plt.close(fig)

    print(f"Experiment saved → {exp_dir}")